In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import vit_b_16

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
data_dir = "/kaggle/input/datasets/afridirahman/brain-stroke-ct-image-dataset/Brain_Data_Organised"

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])


In [ ]:
dataset = datasets.ImageFolder(data_dir, transform=transform)

print("Classes:", dataset.classes)

labels = np.array([dataset.samples[i][1] for i in range(len(dataset))])
print("Class distribution:", np.bincount(labels))


In [ ]:
train_idx, temp_idx = train_test_split(
    np.arange(len(labels)),
    test_size=0.3,
    stratify=labels,
    random_state=42
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=labels[temp_idx],
    random_state=42
)

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)
test_dataset = Subset(dataset, test_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))


In [ ]:
class ViT_Model(nn.Module):
    def __init__(self):
        super(ViT_Model, self).__init__()

        self.backbone = vit_b_16(weights="IMAGENET1K_V1")

        num_features = self.backbone.heads.head.in_features

        self.backbone.heads.head = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, 1)
        )

    def forward(self, x):
        return self.backbone(x)


model = ViT_Model().to(device)


In [ ]:
num_normal = np.sum(labels == 0)
num_stroke = np.sum(labels == 1)

pos_weight = torch.tensor([num_normal / num_stroke]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=3e-5)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5
)


In [ ]:
history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": []
}

epochs = 20

for epoch in range(epochs):

    # TRAIN
    model.train()
    train_loss = 0
    train_preds, train_labels = [], []

    for images, labels_batch in train_loader:

        images = images.to(device)
        labels_batch = labels_batch.float().unsqueeze(1).to(device)

        optimizer.zero_grad()

        outputs = model(images)
        outputs = torch.clamp(outputs, -50, 50)

        loss = criterion(outputs, labels_batch)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()

        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()

        train_preds.extend(preds.cpu().numpy().ravel())
        train_labels.extend(labels_batch.cpu().numpy().ravel())

    train_acc = accuracy_score(train_labels, train_preds)

    # VALIDATION
    model.eval()
    val_loss = 0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for images, labels_batch in val_loader:

            images = images.to(device)
            labels_batch = labels_batch.float().unsqueeze(1).to(device)

            outputs = model(images)
            outputs = torch.clamp(outputs, -50, 50)

            loss = criterion(outputs, labels_batch)
            val_loss += loss.item()

            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()

            val_preds.extend(preds.cpu().numpy().ravel())
            val_labels.extend(labels_batch.cpu().numpy().ravel())

    val_acc = accuracy_score(val_labels, val_preds)

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss/len(train_loader))
    history["val_loss"].append(val_loss/len(val_loader))
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {train_loss/len(train_loader):.4f} | "
          f"Val Loss: {val_loss/len(val_loader):.4f}")


In [ ]:
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels_batch in test_loader:

        images = images.to(device)

        outputs = model(images)
        outputs = torch.clamp(outputs, -50, 50)

        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()

        all_preds.extend(preds.cpu().numpy().ravel())
        all_labels.extend(labels_batch.numpy())
        all_probs.extend(probs.cpu().numpy().ravel())

print("\nCLASSIFICATION REPORT:\n")
print(classification_report(all_labels, all_preds, target_names=["Normal","Stroke"]))

print("Accuracy:", accuracy_score(all_labels, all_preds))
print("Precision:", precision_score(all_labels, all_preds))
print("Recall:", recall_score(all_labels, all_preds))
print("F1 Score:", f1_score(all_labels, all_preds))
print("AUC:", roc_auc_score(all_labels, all_probs))


In [ ]:
cm = confusion_matrix(all_labels, all_preds)

sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=["Normal","Stroke"],
            yticklabels=["Normal","Stroke"],
            cmap="Blues")

plt.title("Confusion Matrix")
plt.show()


In [ ]:
### fpr, tpr, _ = roc_curve(all_labels, all_probs)

plt.plot(fpr, tpr)
plt.plot([0,1],[0,1],'--')
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()
